In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from numpy.linalg import eigh
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import squareform
import warnings
warnings.filterwarnings("ignore")
import os
os.getcwd()

'/content'

## Run HRP Backtest — Marčenko–Pastur Denoised Covariance

**Configuration:**
- Training window: 252 × 4 = 1,008 days
- Holding period: 21 days
- Covariance: Marčenko–Pastur denoised

In [ ]:
equity_hrp_mp, weights_mp, assets_mp, dates_mp = rolling_backtest_hrp(
    df,
    train_window=252*4,
    holding_period=21,
    cov_method="mp"
)

plot_equity_with_assets(equity_hrp_mp, dates_mp, assets_mp, title="HRP (MP Denoised)")

## Step 12: Performance Evaluation

In [ ]:
def performance_metrics(equity):
    """
    Compute key performance metrics for an equity curve.
    Assumes daily returns and 252 trading days/year. Rf = 0.
    """
    equity = np.asarray(equity)
    total_return = (equity[-1] / equity[0] - 1) * 100
    log_returns = np.diff(np.log(equity))
    ann_return = np.mean(log_returns) * 252 * 100
    ann_vol = np.std(log_returns) * np.sqrt(252) * 100
    sharpe = ann_return / ann_vol if ann_vol != 0 else np.nan
    downside = log_returns[log_returns < 0]
    down_std = np.std(downside) * np.sqrt(252) * 100 if len(downside) > 0 else np.nan
    sortino = ann_return / down_std if not np.isnan(down_std) else np.nan
    info = sharpe
    peak = np.maximum.accumulate(equity)
    drawdown = (equity - peak) / peak * 100
    max_dd = np.min(drawdown)
    calmar = ann_return / abs(max_dd) if max_dd != 0 else np.nan
    return {
        'Total Return (%)': round(total_return, 2),
        'Annualized Return (%)': round(ann_return, 2),
        'Annualized Volatility (%)': round(ann_vol, 2),
        'Sharpe Ratio': round(sharpe, 2),
        'Sortino Ratio': round(sortino, 2),
        'Information Ratio': round(info, 2),
        'Max Drawdown (%)': round(max_dd, 2),
        'Calmar Ratio': round(calmar, 2)
    }

print("═" * 50)
print("HRP Performance (Marčenko–Pastur Denoised)")
print("═" * 50)
metrics = performance_metrics(equity_hrp_mp)
for k, v in metrics.items():
    print(f"  {k:30s} {v}")

## Covariance Estimator Comparison

Run HRP with all three estimators for head-to-head comparison.

In [ ]:
# ── Raw sample covariance ──
equity_hrp_raw, weights_raw, assets_raw, dates_raw = rolling_backtest_hrp(
    df, train_window=252*4, holding_period=21, cov_method="raw"
)

# ── Ledoit-Wolf shrinkage ──
equity_hrp_lw, weights_lw, assets_lw, dates_lw = rolling_backtest_hrp(
    df, train_window=252*4, holding_period=21, cov_method="shrink"
)

# ── Comparison table ──
print("\n" + "═" * 65)
print("HRP HEAD-TO-HEAD: Covariance Estimator Comparison")
print("═" * 65)

results = {}
for name, eq in [("Raw Sample", equity_hrp_raw),
                  ("Ledoit-Wolf", equity_hrp_lw),
                  ("MP Denoised", equity_hrp_mp)]:
    m = performance_metrics(eq)
    results[name] = m

df_results = pd.DataFrame(results).T
print(df_results.to_string())

## Equity Curve Comparison — All Three Estimators

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(equity_hrp_raw, label="HRP (Raw Sample)", alpha=0.8, linewidth=1.2)
ax.plot(equity_hrp_lw, label="HRP (Ledoit-Wolf)", alpha=0.8, linewidth=1.2)
ax.plot(equity_hrp_mp, label="HRP (MP Denoised)", alpha=0.8, linewidth=1.5)

ax.set_title("HRP Equity Curves — Covariance Estimator Comparison (2020–2025)", fontsize=14)
ax.set_xlabel("Trading Days (from start of OOS)")
ax.set_ylabel("Equity (starting at 1.0)")
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Weight Distribution Analysis

In [ ]:
# Analyze weight concentration for last rebalance
last_w = weights_mp[-1]
print(f"Last rebalance — MP Denoised HRP weights:")
print(f"  Min weight:   {last_w.min():.4f}")
print(f"  Max weight:   {last_w.max():.4f}")
print(f"  Mean weight:  {last_w.mean():.4f}")
print(f"  # assets > 1%: {np.sum(last_w > 0.01)}")
print(f"  # assets > 0:  {np.sum(last_w > 1e-6)}")
print(f"  Sum:           {last_w.sum():.6f}")
print(f"  Any negative?  {np.any(last_w < 0)}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(last_w, bins=40, edgecolor='black', alpha=0.7, color='#1E2761')
axes[0].set_title("Weight Distribution (Last Rebalance)", fontsize=12)
axes[0].set_xlabel("Weight")
axes[0].set_ylabel("Count")
axes[0].axvline(last_w.mean(), color='red', linestyle='--', label=f'Mean={last_w.mean():.4f}')
axes[0].legend()

# Weight evolution over time
max_weights = [w.max() for w in weights_mp]
min_weights = [w.min() for w in weights_mp]
axes[1].plot(max_weights, label="Max weight", color='#4A6CF7')
axes[1].plot(min_weights, label="Min weight", color='#C0392B')
axes[1].set_title("Weight Extremes Over Time", fontsize=12)
axes[1].set_xlabel("Rebalance Period")
axes[1].set_ylabel("Weight")
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Benchmark Comparison — Rolling Correlation with S&P 500

Same analysis as PCA pipeline: 252-day rolling correlation.

In [ ]:
def align_strategy_benchmark_simple(strategy_equity, df):
    """Align strategy returns with S&P 500 from local file."""
    try:
        sp = pd.read_excel("SPX_10_RISK.xls")
        sp['Date'] = pd.to_datetime(sp['Date'])
        sp = sp[['Date', 'Price']]
        sp.rename(columns={'Price': 'Close'}, inplace=True)
    except:
        print("SPX_10_RISK.xls not found. Skipping benchmark comparison.")
        return None

    sp['returns'] = np.log(sp['Close'] / sp['Close'].shift(1))
    sp.dropna(inplace=True)

    strat_returns = np.diff(strategy_equity) / strategy_equity[:-1]
    n = min(len(strat_returns), len(sp))

    return pd.DataFrame({
        "strategy": strat_returns[:n],
        "benchmark": sp['returns'].values[:n]
    })

returns_df = align_strategy_benchmark_simple(equity_hrp_mp, df)

if returns_df is not None:
    roll_corr = returns_df['strategy'].rolling(252).corr(returns_df['benchmark'])

    plt.figure(figsize=(12, 6))
    plt.plot(roll_corr, color='#1E2761', linewidth=1.2)
    plt.axhline(0, linestyle='--', color='gray', alpha=0.5)
    plt.title("HRP (MP) — Rolling 252-Day Correlation with S&P 500", fontsize=14)
    plt.xlabel("Time")
    plt.ylabel("Correlation")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("Benchmark file not available — run with SPX_10_RISK.xls in directory.")